# LeetCode 829 — Consecutive Numbers Sum bằng Python

- Link: https://leetcode.com/problems/consecutive-numbers-sum/
- Độ khó: **Hard**
- Chủ đề: Toán (số học), Tổng cấp số cộng, Ước số lẻ, Phân tích thừa số nguyên tố, Hai con trỏ / cửa sổ trượt

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def consecutiveNumbersSum(self, n: int) -> int:
```

## 1. Giới thiệu & mục tiêu học

Có bao nhiêu cách viết số nguyên dương `n` thành tổng của **một hoặc nhiều số nguyên dương liên tiếp**? Ví dụ `15 = 15 = 7+8 = 4+5+6 = 1+2+3+4+5` → 4 cách.

Với `n` tới `10⁹`, mọi cách duyệt theo `n` đều quá chậm. Bài này dạy cách **biến đổi đại số** để giảm không gian tìm kiếm: cố định số số hạng `k`, điều kiện tồn tại trở thành một phép chia hết, và `k` chỉ cần chạy tới `√(2n)`. Đi xa hơn, đáp án chính là **số ước lẻ** của `n` — một kết quả số học rất đẹp.

Sau notebook này bạn sẽ:

1. Viết brute force và cửa sổ trượt (hai con trỏ) cho tổng dãy liên tiếp.
2. Dùng công thức tổng cấp số cộng `x + (x+1) + ... + (x+k−1) = kx + k(k−1)/2`.
3. Suy ra điều kiện `(n − k(k−1)/2) % k == 0` và chặn `k(k+1)/2 <= n`.
4. Chứng minh đáp án = số ước lẻ của `n`, tính bằng phân tích thừa số nguyên tố.
5. Tránh các lỗi về số học: chia nguyên, biên vòng lặp, `k = 1`.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho số nguyên `n`. Đếm số dãy gồm **một hoặc nhiều** số nguyên **dương liên tiếp** có tổng bằng `n`.

**Ví dụ 1**

```text
Input: n = 5
Output: 2
Giải thích: 5 = 5 = 2 + 3
```

**Ví dụ 2**

```text
Input: n = 9
Output: 3
Giải thích: 9 = 9 = 4 + 5 = 2 + 3 + 4
```

**Ví dụ 3**

```text
Input: n = 15
Output: 4
Giải thích: 15 = 15 = 8 + 7 = 4 + 5 + 6 = 1 + 2 + 3 + 4 + 5
```

**Ràng buộc**

- `1 <= n <= 10⁹`

In [1]:
from typing import List

EXAMPLES = [
    (5, 2),
    (9, 3),
    (15, 4),
    (1, 1),
    (2, 1),     # chỉ "2"
    (8, 1),     # lũy thừa của 2: chỉ chính nó
    (3, 2),     # 3 = 1 + 2
]
for n, exp in EXAMPLES:
    print("n =", n, "-> kỳ vọng", exp)

n = 5 -> kỳ vọng 2
n = 9 -> kỳ vọng 3
n = 15 -> kỳ vọng 4
n = 1 -> kỳ vọng 1
n = 2 -> kỳ vọng 1
n = 8 -> kỳ vọng 1
n = 3 -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Tổng cấp số cộng

`x + (x+1) + ... + (x+k−1) = k·x + (0 + 1 + ... + (k−1)) = k·x + k(k−1)/2`.

In [2]:
def seq_sum(x, k):
    return k * x + k * (k - 1) // 2

print(seq_sum(4, 3), 4 + 5 + 6)
print(seq_sum(1, 5), sum(range(1, 6)))

15 15
15 15


### 3.2. Phép chia nguyên và chia hết

`a // b` chia lấy phần nguyên, `a % b` lấy dư. `a % b == 0` ⇔ `b` chia hết `a`. Với số dương, Python không có bất ngờ; tránh dùng `/` (số thực, mất chính xác khi lớn).

In [3]:
print(17 // 5, 17 % 5, 15 % 5 == 0)
big = 10 ** 18 + 1
print(big // 3, int(big / 3))      # / cho số thực, sai lệch khi lớn

3 2 True
333333333333333333 333333333333333312


### 3.3. Căn bậc hai nguyên `math.isqrt`

`math.isqrt(m)` = phần nguyên của `√m`, chính xác cho số lớn. Dùng để chặn vòng lặp.

In [4]:
import math
print(math.isqrt(2 * 10 ** 9), int(math.sqrt(2 * 10 ** 9)))

44721 44721


### 3.4. Ước số và ước lẻ

Ước của `n` đi theo cặp `(d, n/d)`, nên chỉ cần duyệt `d <= √n`. Ước lẻ: bỏ hết thừa số 2 khỏi `n` rồi đếm ước của phần còn lại.

In [5]:
def divisors(n):
    small, large = [], []
    d = 1
    while d * d <= n:
        if n % d == 0:
            small.append(d)
            if d != n // d:
                large.append(n // d)
        d += 1
    return small + large[::-1]

print(divisors(36), [d for d in divisors(36) if d % 2])
print(divisors(15))

[1, 2, 3, 4, 6, 9, 12, 18, 36] [1, 3, 9]
[1, 3, 5, 15]


### 3.5. Phân tích thừa số nguyên tố và đếm ước

Nếu `m = p₁^a₁ · p₂^a₂ · ...` thì số ước của `m` là `(a₁ + 1)(a₂ + 1)...`.

In [6]:
def factorize(m):
    f = {}
    p = 2
    while p * p <= m:
        while m % p == 0:
            f[p] = f.get(p, 0) + 1
            m //= p
        p += 1
    if m > 1:
        f[m] = f.get(m, 0) + 1
    return f

def count_divisors(m):
    res = 1
    for a in factorize(m).values():
        res *= a + 1
    return res

print(factorize(360), count_divisors(360), len(divisors(360)))

{2: 3, 3: 2, 5: 1} 24 24


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Liệt kê tay `n = 15`

| k (số số hạng) | x (số đầu) | dãy |
|---|---|---|
| 1 | 15 | 15 |
| 2 | 7 | 7 + 8 |
| 3 | 4 | 4 + 5 + 6 |
| 4 | — | `15 − 6 = 9`, 9 không chia hết cho 4 |
| 5 | 1 | 1 + 2 + 3 + 4 + 5 |
| 6 | — | `6·7/2 = 21 > 15`, dừng |

→ 4 cách.

### 4.2. Điều kiện cho từng `k`

`n = k·x + k(k−1)/2` với `x >= 1` ⇔ `x = (n − k(k−1)/2) / k` là số nguyên dương ⇔

- `n − k(k−1)/2 > 0` ⇔ `k(k+1)/2 <= n` (vì `x >= 1`), và
- `(n − k(k−1)/2) % k == 0`.

Vì `k(k+1)/2 <= n` nên `k < √(2n)` — với `n = 10⁹` chỉ khoảng 44 721 giá trị `k`.

### 4.3. Quan sát về ước lẻ

`n = 15 = 3·5` có ước lẻ `1, 3, 5, 15` — đúng 4. `n = 9`: ước lẻ `1, 3, 9` — 3. `n = 8`: ước lẻ chỉ `1` → 1. Trùng hợp? Không — mục 10 sẽ chứng minh.

In [7]:
def list_ways(n):
    ways = []
    k = 1
    while k * (k + 1) // 2 <= n:
        rest = n - k * (k - 1) // 2
        if rest % k == 0:
            x = rest // k
            ways.append(list(range(x, x + k)))
        k += 1
    return ways

for n in (9, 15, 8):
    print(n, list_ways(n), "| ước lẻ:", [d for d in divisors(n) if d % 2])

9 [[9], [4, 5], [2, 3, 4]] | ước lẻ: [1, 3, 9]
15 [[15], [7, 8], [4, 5, 6], [1, 2, 3, 4, 5]] | ước lẻ: [1, 3, 5, 15]
8 [[8]] | ước lẻ: [1]


### 4.4. Thêm ví dụ tính tay: n = 30 và n = 32

**n = 30 = 2 · 3 · 5.** Phần lẻ là `15`, ước lẻ `1, 3, 5, 15` → dự đoán **4 cách**. Kiểm tra theo `k`:

| k | `k(k−1)/2` | `n − k(k−1)/2` | chia hết cho k? | x | dãy |
|---|---|---|---|---|---|
| 1 | 0 | 30 | ✔ | 30 | 30 |
| 2 | 1 | 29 | ✗ | — | — |
| 3 | 3 | 27 | ✔ | 9 | 9+10+11 |
| 4 | 6 | 24 | ✔ | 6 | 6+7+8+9 |
| 5 | 10 | 20 | ✔ | 4 | 4+5+6+7+8 |
| 6 | 15 | 15 | ✗ | — | — |
| 7 | 21 | 9 | ✗ (`9 % 7 ≠ 0`) | — | — |
| 8 | 28 | 2 | dừng vì `2 < 8` (x < 1) | — | — |

→ 4 cách ✔.

**n = 32 = 2⁵.** Phần lẻ là `1` → chỉ **1 cách** (chính 32). Thử `k = 2`: `31` lẻ ✗; `k = 4`: `26 % 4 ≠ 0` ✗; ... Lũy thừa của 2 không bao giờ là tổng của ≥ 2 số dương liên tiếp.

**Quan sát tính chẵn lẻ:** với `k` lẻ, điều kiện là `k | n` (vì `k(k−1)/2` chia hết cho `k` khi `k` lẻ). Với `k` chẵn, điều kiện là `n ≡ k/2 (mod k)`. Đây là lý do ước lẻ xuất hiện.

In [8]:
for n in (30, 32):
    print(n, "->", list_ways(n), "| ước lẻ:", [d for d in divisors(n) if d % 2])

# Kiểm chứng quan sát tính chẵn lẻ của k
for n in range(1, 200):
    for k in range(1, 30):
        if k * (k + 1) // 2 > n:
            break
        cond = (n - k * (k - 1) // 2) % k == 0
        alt = (n % k == 0) if k % 2 else (n % k == k // 2)
        assert cond == alt
print("Điều kiện theo tính chẵn lẻ của k: đúng với mọi n < 200")

30 -> [[30], [9, 10, 11], [6, 7, 8, 9], [4, 5, 6, 7, 8]] | ước lẻ: [1, 3, 5, 15]
32 -> [[32]] | ước lẻ: [1]
Điều kiện theo tính chẵn lẻ của k: đúng với mọi n < 200


## 5. Cách 1 — Brute Force: thử mọi số bắt đầu

**Ý tưởng.** Với mỗi số đầu `x = 1..n`, cộng dần `x, x+1, ...` cho tới khi tổng ≥ `n`; nếu tổng đúng bằng `n` thì đếm.

In [9]:
def consec_brute(n):
    count = 0
    for x in range(1, n + 1):
        total, y = 0, x
        while total < n:
            total += y
            y += 1
        if total == n:
            count += 1
    return count

for n, exp in EXAMPLES:
    assert consec_brute(n) == exp
print("Brute force: OK", [consec_brute(n) for n in range(1, 21)])

Brute force: OK [1, 1, 2, 1, 2, 2, 2, 1, 3, 2, 2, 2, 2, 2, 4, 1, 2, 3, 2, 2]


**Độ phức tạp.** Với `x` nhỏ, vòng trong chạy khoảng `√(2n)` bước; với `x` lớn chỉ 1–2 bước. Tổng cỡ `O(n)` đến `O(n√n)` tuỳ cách đếm — thực tế `O(n)` đã quá chậm với `n = 10⁹`.

**Nút thắt.** Ta duyệt theo **số bắt đầu** (`n` giá trị). Nhưng số **số hạng** `k` chỉ có `O(√n)` giá trị khả dĩ, và với mỗi `k` thì số bắt đầu được xác định duy nhất!

## 6. Bước đột phá

**Insight 1 — Cố định số số hạng.** Với mỗi `k`, có **nhiều nhất một** dãy `k` số liên tiếp có tổng `n` (vì tổng tăng ngặt theo `x`). Dãy đó tồn tại ⇔ `x = (n − k(k−1)/2)/k` nguyên dương.

**Insight 2 — Chặn trên của `k`.** Dãy nhỏ nhất có `k` số là `1 + 2 + ... + k = k(k+1)/2`. Vậy chỉ cần `k(k+1)/2 <= n`, tức `k = O(√n)`.

**Insight 3 — Ước lẻ.** `2n = k·(2x + k − 1)`. Hai thừa số `k` và `2x + k − 1` có **tính chẵn lẻ khác nhau** (tổng của chúng `2x + 2k − 1` lẻ), và `2x + k − 1 > k`. Mỗi cách chọn ước lẻ `d` của `n` cho đúng một cặp như vậy → đáp án = số ước lẻ của `n`.

**Quy tắc nếu…thì:**
- *Nếu* `n` là lũy thừa của 2 *thì* đáp án = 1.
- *Nếu* `n` lẻ *thì* đáp án = số ước của `n` (mọi ước đều lẻ).

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Cửa sổ trượt (hai con trỏ), `O(n)`

Duy trì đoạn `[lo, hi]` và tổng `s`. Nếu `s < n` thì mở rộng `hi`, nếu `s > n` thì thu hẹp `lo`, nếu `s == n` thì đếm rồi thu hẹp. Mỗi con trỏ chỉ tăng → `O(n)`.

In [10]:
def consec_window(n):
    count = 0
    lo, s = 1, 0
    for hi in range(1, n + 1):
        s += hi
        while s > n:
            s -= lo
            lo += 1
        if s == n:
            count += 1
    return count

for n, exp in EXAMPLES:
    assert consec_window(n) == exp
print("Cửa sổ trượt: OK")

Cửa sổ trượt: OK


Code "nộp" cho cách 2 (TLE với `n = 10⁹`, nhưng đúng):

In [11]:
class SolutionWindow:
    def consecutiveNumbersSum(self, n: int) -> int:
        count, lo, s = 0, 1, 0
        for hi in range(1, n + 1):
            s += hi
            while s > n:
                s -= lo
                lo += 1
            if s == n:
                count += 1
        return count

for n, exp in EXAMPLES:
    assert SolutionWindow().consecutiveNumbersSum(n) == exp
print("SolutionWindow: OK")

SolutionWindow: OK


**Độ phức tạp cách 2:** `O(n)` thời gian, `O(1)` bộ nhớ.

### 7.2. Cách 3 — Duyệt số số hạng `k`, `O(√n)`

In [12]:
def consec_k(n):
    count = 0
    k = 1
    while k * (k + 1) // 2 <= n:
        if (n - k * (k - 1) // 2) % k == 0:
            count += 1
        k += 1
    return count

for n, exp in EXAMPLES:
    assert consec_k(n) == exp
print("Duyệt k: OK", consec_k(10 ** 9))

Duyệt k: OK 10


Biến thể gọn hơn: trừ dần. Đặt `m = n`; với `k = 1, 2, ...`: `m -= (k − 1)` (khi đó `m = n − k(k−1)/2`), kiểm tra `m % k == 0`, dừng khi `m <= 0`... cụ thể `m` phải ≥ `k` để `x >= 1`.

Code nộp LeetCode cho cách 3:

In [13]:
class SolutionK:
    def consecutiveNumbersSum(self, n: int) -> int:
        count = 0
        k = 1
        m = n                      # m = n - k(k-1)/2
        while m >= k:              # x = m / k >= 1
            if m % k == 0:
                count += 1
            m -= k                 # chuyển sang k+1: trừ thêm k
            k += 1
        return count

for n, exp in EXAMPLES:
    assert SolutionK().consecutiveNumbersSum(n) == exp
print("SolutionK: OK")

SolutionK: OK


Kiểm tra: với `k`, `m = n − (0 + 1 + ... + (k−1))`. Sang `k+1` cần trừ thêm `k` → `m -= k` rồi `k += 1`. Điều kiện `m >= k` ⇔ `x = m/k >= 1` (với điều kiện chia hết) ⇔ `n − k(k−1)/2 >= k` ⇔ `k(k+1)/2 <= n` ✔.

**Độ phức tạp cách 3:** `O(√n)` thời gian (≈ 44 721 vòng với `n = 10⁹`), `O(1)` bộ nhớ.

### 7.3. Cách 4 — Đếm ước lẻ bằng duyệt `d <= √m`

Bỏ hết thừa số 2: `m = n / 2^a` (lẻ). Đếm ước của `m` bằng cách duyệt `d` lẻ tới `√m`.

In [14]:
def consec_odd_divisors(n):
    while n % 2 == 0:
        n //= 2
    count = 0
    d = 1
    while d * d <= n:
        if n % d == 0:
            count += 1 if d * d == n else 2
        d += 2                     # n lẻ → chỉ có ước lẻ
    return count

for n, exp in EXAMPLES:
    assert consec_odd_divisors(n) == exp
print("Đếm ước lẻ: OK", consec_odd_divisors(10 ** 9))

Đếm ước lẻ: OK 10


Code nộp LeetCode cho cách 4:

In [15]:
class SolutionOddDiv:
    def consecutiveNumbersSum(self, n: int) -> int:
        while n % 2 == 0:
            n //= 2
        count, d = 0, 1
        while d * d <= n:
            if n % d == 0:
                count += 1 if d * d == n else 2
            d += 2
        return count

for n, exp in EXAMPLES:
    assert SolutionOddDiv().consecutiveNumbersSum(n) == exp
print("SolutionOddDiv: OK")

SolutionOddDiv: OK


**Độ phức tạp cách 4:** `O(√m)` với `m` = phần lẻ của `n`, bước nhảy 2 nên khoảng `√n / 2` vòng.

### 7.4. Cách 5 — Phân tích thừa số nguyên tố

Số ước lẻ = `∏ (aᵢ + 1)` trên các thừa số nguyên tố **lẻ** `pᵢ^aᵢ` của `n`. Vòng phân tích dừng sớm khi `m` đã nhỏ — với nhiều `n` nhanh hơn cách 4 rất nhiều.

In [16]:
def consec_factor(n):
    while n % 2 == 0:
        n //= 2
    res = 1
    p = 3
    while p * p <= n:
        a = 0
        while n % p == 0:
            n //= p
            a += 1
        res *= a + 1
        p += 2
    if n > 1:                      # thừa số nguyên tố lớn còn lại
        res *= 2
    return res

for n, exp in EXAMPLES:
    assert consec_factor(n) == exp
print("Phân tích thừa số: OK", consec_factor(10 ** 9), factorize(10 ** 9))

Phân tích thừa số: OK 10 {2: 9, 5: 9}


Code nộp LeetCode cho cách 5:

In [17]:
class SolutionFactor:
    def consecutiveNumbersSum(self, n: int) -> int:
        while n % 2 == 0:
            n //= 2
        res, p = 1, 3
        while p * p <= n:
            a = 0
            while n % p == 0:
                n //= p
                a += 1
            res *= a + 1
            p += 2
        if n > 1:
            res *= 2
        return res

for n, exp in EXAMPLES:
    assert SolutionFactor().consecutiveNumbersSum(n) == exp
print("SolutionFactor: OK")

SolutionFactor: OK


**Độ phức tạp cách 5:** xấu nhất `O(√n)` (khi phần lẻ là số nguyên tố lớn), thường nhanh hơn nhiều vì `n` co lại trong lúc chia.

## 8. TRACE từng bước

### 8.1. Trace cách 3 với `n = 15`

In [18]:
def trace_k(n):
    count, k, m = 0, 1, n
    print(f"{'k':>3} {'m=n-k(k-1)/2':>14} {'m % k':>6} {'x':>4} {'đếm':>4}")
    while m >= k:
        ok = m % k == 0
        if ok:
            count += 1
        print(f"{k:>3} {m:>14} {m % k:>6} {m // k if ok else '-':>4} {count:>4}")
        m -= k
        k += 1
    print(f"dừng ở k={k}: m={m} < k")
    return count

trace_k(15)

  k   m=n-k(k-1)/2  m % k    x  đếm
  1             15      0   15    1
  2             14      0    7    2
  3             12      0    4    3
  4              9      1    -    3
  5              5      0    1    4
dừng ở k=6: m=0 < k


4

Bảng trạng thái:

| k | m | m % k | x | count |
|---|---|---|---|---|
| 1 | 15 | 0 | 15 | 1 |
| 2 | 14 | 0 | 7 | 2 |
| 3 | 12 | 0 | 4 | 3 |
| 4 | 9 | 1 | – | 3 |
| 5 | 5 | 0 | 1 | 4 |
| 6 | 0 | — | dừng (`m < k`) | 4 |

### 8.2. Trace cách 5 (phân tích thừa số) với `n = 360 = 2³ · 3² · 5`

In [19]:
def trace_factor(n):
    orig = n
    a2 = 0
    while n % 2 == 0:
        n //= 2
        a2 += 1
    print(f"{orig} = 2^{a2} * {n}  (bỏ thừa số 2)")
    res, p = 1, 3
    while p * p <= n:
        a = 0
        while n % p == 0:
            n //= p
            a += 1
        if a:
            print(f"  p={p}: số mũ {a} -> nhân ({a}+1); res = {res * (a + 1)}; còn lại {n}")
        res *= a + 1
        p += 2
    if n > 1:
        res *= 2
        print(f"  thừa số nguyên tố lớn còn lại {n} -> nhân 2; res = {res}")
    print("số ước lẻ =", res)
    return res

trace_factor(360)
print("ước lẻ của 360:", [d for d in divisors(360) if d % 2])

360 = 2^3 * 45  (bỏ thừa số 2)
  p=3: số mũ 2 -> nhân (2+1); res = 3; còn lại 5
  thừa số nguyên tố lớn còn lại 5 -> nhân 2; res = 6
số ước lẻ = 6
ước lẻ của 360: [1, 3, 5, 9, 15, 45]


### 8.3. Trace cửa sổ trượt với `n = 9`

In [20]:
def trace_window(n):
    count, lo, s = 0, 1, 0
    for hi in range(1, n + 1):
        s += hi
        while s > n:
            s -= lo
            lo += 1
        hit = s == n
        count += hit
        print(f"hi={hi}: đoạn [{lo}..{hi}] tổng {s}{'  <-- bằng n' if hit else ''}")
    return count

trace_window(9)

hi=1: đoạn [1..1] tổng 1
hi=2: đoạn [1..2] tổng 3
hi=3: đoạn [1..3] tổng 6
hi=4: đoạn [2..4] tổng 9  <-- bằng n
hi=5: đoạn [4..5] tổng 9  <-- bằng n
hi=6: đoạn [6..6] tổng 6
hi=7: đoạn [7..7] tổng 7
hi=8: đoạn [8..8] tổng 8
hi=9: đoạn [9..9] tổng 9  <-- bằng n


3

## 9. Code nộp LeetCode

Chọn cách 3 (duyệt `k`): ngắn, dễ giải thích, `O(√n)` — đủ nhanh cho `n = 10⁹`.

In [21]:
from typing import List

class Solution:
    def consecutiveNumbersSum(self, n: int) -> int:
        count = 0
        k = 1
        m = n                       # m = n - k(k-1)/2 = k * x
        while m >= k:               # cần x = m / k >= 1
            if m % k == 0:          # x nguyên → có đúng một dãy k số
                count += 1
            m -= k                  # chuẩn bị cho k + 1
            k += 1
        return count

for n, exp in EXAMPLES:
    assert Solution().consecutiveNumbersSum(n) == exp
print("Solution: OK")

Solution: OK


**Giải thích từng dòng của `Solution`:**

| dòng | ý nghĩa |
|---|---|
| `count = 0` | số dãy tìm được |
| `k = 1`, `m = n` | bắt đầu với dãy 1 số: `m = n − 0` |
| `while m >= k` | còn có thể có `x = m / k >= 1` |
| `if m % k == 0` | `x` nguyên → đúng một dãy `k` số |
| `m -= k` | `n − (k+1)k/2 = (n − k(k−1)/2) − k` |
| `k += 1` | chuyển sang số số hạng tiếp theo |

Số vòng lặp tối đa ≈ `√(2n)` ≈ 44 721 với `n = 10⁹` — nhỏ hơn nhiều so với `n`.

In [22]:
import math
for n in (10, 10 ** 3, 10 ** 6, 10 ** 9):
    loops, k, m = 0, 1, n
    while m >= k:
        loops += 1; m -= k; k += 1
    print(f"n = {n:>10}: {loops:>6} vòng lặp, √(2n) ≈ {math.isqrt(2 * n)}")

n =         10:      4 vòng lặp, √(2n) ≈ 4
n =       1000:     44 vòng lặp, √(2n) ≈ 44
n =    1000000:   1413 vòng lặp, √(2n) ≈ 1414
n = 1000000000:  44720 vòng lặp, √(2n) ≈ 44721


## 10. Vì sao thuật toán đúng

**Cách 3.** Một dãy `k >= 1` số nguyên dương liên tiếp bắt đầu từ `x >= 1` có tổng `S(x, k) = kx + k(k−1)/2`. Với `k` cố định, `S` tăng ngặt theo `x`, nên có nhiều nhất một `x` cho tổng `n`: `x = (n − k(k−1)/2)/k`. Dãy hợp lệ ⇔ `x` là số nguyên `>= 1` ⇔ `k | (n − k(k−1)/2)` và `n − k(k−1)/2 >= k`. Vòng lặp kiểm tra đúng các `k` thoả bất đẳng thức (và khi bất đẳng thức sai với `k` thì cũng sai với mọi `k' > k`, vì `k(k+1)/2` tăng). Các dãy với `k` khác nhau là khác nhau → đếm đúng.

**Cách 4, 5 (ước lẻ).** Từ `2n = k(2x + k − 1)`, đặt `A = k`, `B = 2x + k − 1`. Ta có:

- `A + B = 2x + 2k − 1` lẻ ⇒ `A`, `B` khác tính chẵn lẻ.
- `B − A = 2x − 1 >= 1` ⇒ `B > A`.

Ngược lại, mọi phân tích `2n = A·B` với `A < B`, khác tính chẵn lẻ, cho `k = A`, `x = (B − A + 1)/2` nguyên dương. Vậy số cách = số cặp `(A, B)` như vậy.

Viết `n = 2ᵃ · m` với `m` lẻ. Trong `2n = 2^{a+1}·m`, thừa số chẵn trong cặp phải chứa **toàn bộ** `2^{a+1}` (thừa số kia lẻ). Vậy cặp được xác định bởi ước lẻ `d` của `m`: một thừa số là `d`, thừa số kia là `2^{a+1}·m/d`; hai số này khác nhau (một chẵn, một lẻ) nên có đúng một cách gán `A < B`. → song ánh với tập ước lẻ của `n`. ∎

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(n√n)` | `O(1)` |
| 2. Cửa sổ trượt | `O(n)` | `O(1)` |
| 3. Duyệt `k` | `O(√n)` | `O(1)` |
| 4. Đếm ước lẻ | `O(√n)` (bước 2) | `O(1)` |
| 5. Phân tích thừa số | `O(√n)` xấu nhất, thường nhanh hơn | `O(1)` |

In [23]:
import time, random

for name, fn in [("k", consec_k), ("SolutionK", lambda n: SolutionK().consecutiveNumbersSum(n)),
                 ("odd_div", consec_odd_divisors), ("factor", consec_factor),
                 ("Solution", lambda n: Solution().consecutiveNumbersSum(n))]:
    t0 = time.perf_counter()
    res = [fn(x) for x in (10 ** 9, 999999937, 735134400)]
    print(f"{name:10}: {res}  ({(time.perf_counter() - t0) * 1000:.0f} ms)")

t0 = time.perf_counter(); consec_window(10 ** 6); print(f"cửa sổ trượt n=1e6: {(time.perf_counter() - t0) * 1000:.0f} ms")

k         : [10, 2, 192]  (20 ms)


SolutionK : [10, 2, 192]  (9 ms)
odd_div   : [10, 2, 192]  (1 ms)
factor    : [10, 2, 192]  (1 ms)


Solution  : [10, 2, 192]  (10 ms)


cửa sổ trượt n=1e6: 85 ms


## 12. Bộ test

In [24]:
methods = {
    "brute": consec_brute,
    "window": consec_window,
    "k": consec_k,
    "odd_div": consec_odd_divisors,
    "factor": consec_factor,
    "SolutionWindow": lambda n: SolutionWindow().consecutiveNumbersSum(n),
    "SolutionK": lambda n: SolutionK().consecutiveNumbersSum(n),
    "SolutionOddDiv": lambda n: SolutionOddDiv().consecutiveNumbersSum(n),
    "SolutionFactor": lambda n: SolutionFactor().consecutiveNumbersSum(n),
    "Solution": lambda n: Solution().consecutiveNumbersSum(n),
}

# 12.1 Ví dụ + edge cases
edge = EXAMPLES + [(1024, 1), (45, 6), (105, 8), (999, 8)]
for name, fn in methods.items():
    for n, exp in edge:
        assert fn(n) == exp, (name, n, fn(n), exp)
print("Ví dụ + edge cases: OK")

# 12.2 Mọi n nhỏ: đối chiếu brute force
for n in range(1, 1501):
    ref = consec_brute(n)
    for name, fn in methods.items():
        assert fn(n) == ref, (name, n)
print("n = 1..1500: khớp brute force")

# 12.3 n lớn ngẫu nhiên có seed: các cách nhanh khớp nhau
rnd = random.Random(829)
fast = ["k", "odd_div", "factor", "SolutionK", "SolutionOddDiv", "SolutionFactor", "Solution"]
for _ in range(40):
    n = rnd.randint(1, 10 ** 9)
    vals = {name: methods[name](n) for name in fast}
    assert len(set(vals.values())) == 1, (n, vals)
print("40 số lớn ngẫu nhiên: các cách nhanh khớp nhau")

Ví dụ + edge cases: OK


n = 1..1500: khớp brute force


40 số lớn ngẫu nhiên: các cách nhanh khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Cho phép số 0 hoặc số âm trong dãy

**Code sai:**

In [25]:
def wrong_allow_zero(n):
    count, k = 0, 1
    while k * (k - 1) // 2 <= n:                 # SAI: chặn lỏng, cho phép x = 0
        if (n - k * (k - 1) // 2) % k == 0:
            count += 1
        k += 1
    return count

print("n=15  sai:", wrong_allow_zero(15), " đúng:", Solution().consecutiveNumbersSum(15))

n=15  sai: 5  đúng: 4


**Vì sao sai:** với `k = 6`, `n − 15 = 0` chia hết cho 6 → `x = 0`, tức dãy `0 + 1 + 2 + 3 + 4 + 5` — có số 0, không hợp lệ. Mỗi dãy bắt đầu từ 1 lại được đếm thêm một bản "có 0".

**Code đúng:** điều kiện `k(k+1)/2 <= n` (tức `x >= 1`).

### Lỗi 2 — Dùng phép chia số thực

**Code sai:**

In [26]:
def wrong_float(n):
    count, k = 0, 1
    while k * (k + 1) / 2 <= n:
        x = (n - k * (k - 1) / 2) / k
        if x == int(x):                           # SAI tiềm ẩn: so sánh số thực
            count += 1
        k += 1
    return count

n_big, k = 10 ** 17 + 2, 2
x_float = (n_big - k * (k - 1) / 2) / k
print("số thực: x =", x_float, "-> trông như số nguyên?", x_float == int(x_float))
print("số nguyên: (n - 1) % 2 =", (n_big - 1) % 2, "-> thực ra KHÔNG chia hết")
print("với n nhỏ hai cách vẫn khớp:", wrong_float(15), Solution().consecutiveNumbersSum(15))

số thực: x = 5e+16 -> trông như số nguyên? True
số nguyên: (n - 1) % 2 = 1 -> thực ra KHÔNG chia hết
với n nhỏ hai cách vẫn khớp: 4 4


**Vì sao sai:** `float` chỉ có ~15–16 chữ số chính xác. Ở demo, `n = 10¹⁷ + 2`, `k = 2`: `n − 1` lẻ nên không có dãy 2 số, nhưng `float` làm tròn `10¹⁷ + 1` thành `10¹⁷` và báo nhầm là chia hết. Với `n` tới `10⁹` thì chưa lỗi, nhưng đây là thói quen nguy hiểm (biến thể `n` tới `10¹⁸` sẽ sai). **Code đúng:** dùng `//` và `%` trên số nguyên.

### Lỗi 3 — Đếm **mọi** ước thay vì ước **lẻ**

**Code sai:**

In [27]:
def wrong_all_divisors(n):
    return len(divisors(n))                       # SAI

print("n=12  sai:", wrong_all_divisors(12), " đúng:", Solution().consecutiveNumbersSum(12))

n=12  sai: 6  đúng: 2


**Vì sao sai:** `12` có 6 ước nhưng chỉ có 2 cách: `12` và `3 + 4 + 5`. Chỉ ước lẻ mới tương ứng một dãy. **Code đúng:** bỏ thừa số 2 trước khi đếm ước.

### Lỗi 4 — Quên thừa số nguyên tố lớn còn lại

**Code sai:**

In [28]:
def wrong_forget_last_prime(n):
    while n % 2 == 0:
        n //= 2
    res, p = 1, 3
    while p * p <= n:
        a = 0
        while n % p == 0:
            n //= p; a += 1
        res *= a + 1
        p += 2
    return res                                    # SAI: thiếu "if n > 1: res *= 2"

print("n=15  sai:", wrong_forget_last_prime(15), " đúng:", Solution().consecutiveNumbersSum(15))

n=15  sai: 2  đúng: 4


**Vì sao sai:** `15 = 3·5`; sau khi chia 3, còn `5` nhưng `p = 5` có `5·5 > 5` nên vòng dừng — thừa số 5 bị bỏ sót. **Code đúng:** sau vòng, nếu `n > 1` thì đó là một số nguyên tố, nhân thêm 2.

## 14. Mẹo & nhận dạng pattern

- **"Tổng các số liên tiếp"** → công thức cấp số cộng `kx + k(k−1)/2`; cố định **độ dài** thay vì điểm bắt đầu.
- Khi một biến (ở đây `x`) được xác định duy nhất bởi biến kia (`k`), chỉ cần duyệt biến có miền nhỏ hơn.
- Phương trình dạng `A·B = hằng số` với ràng buộc chẵn lẻ → nghĩ tới **đếm ước**.
- Cửa sổ trượt là công cụ chung cho "đoạn con có tổng = target" khi các phần tử **dương**.
- Bài liên quan: **1010**, **1018**, **2443 Sum of Number and Its Reverse**, **1291 Sequential Digits**, **633 Sum of Square Numbers**.

## 15. Biến thể

1. **Liệt kê** các dãy, không chỉ đếm (hàm `list_ways` ở mục 4).
2. **Cho phép số âm và 0** (dãy số nguyên liên tiếp bất kỳ): mỗi dãy dương `[x..x+k−1]` có bản đối xứng `[−x+1..x+k−1]` → đáp án gấp đôi = `2 × (số ước lẻ)`.
3. **Ít nhất 2 số hạng**: trừ 1 (bỏ dãy chỉ gồm `n`).
4. **Dãy cách đều bước `d`** (`x, x+d, x+2d, ...`): `n = kx + d·k(k−1)/2`, vẫn duyệt `k`.
5. **Nhiều truy vấn `n` tới `10⁷`**: sàng ước số / sàng ước nguyên tố nhỏ nhất rồi đếm ước lẻ mỗi truy vấn trong `O(log n)`.

Minh hoạ biến thể 2 và 4:

In [29]:
def ways_integers(n):
    """Dãy số nguyên liên tiếp (có thể âm/0) có tổng n."""
    count = 0
    for x in range(-n, n + 1):
        s, y = 0, x
        while y <= n:
            s += y
            if s == n:
                count += 1
                break
            if s > n and y > 0:
                break
            y += 1
    return count

def ways_step(n, d):
    count, k = 0, 1
    while k + d * k * (k - 1) // 2 <= n:
        if (n - d * k * (k - 1) // 2) % k == 0:
            count += 1
        k += 1
    return count

for n in (5, 9, 15, 16):
    assert ways_integers(n) == 2 * Solution().consecutiveNumbersSum(n)
print("số nguyên bất kỳ:", [ways_integers(n) for n in (5, 9, 15, 16)])
print("bước 2 (các số lẻ/chẵn liên tiếp) cho n=16:", ways_step(16, 2))

số nguyên bất kỳ: [4, 6, 8, 2]
bước 2 (các số lẻ/chẵn liên tiếp) cho n=16: 3


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | thử mọi số đầu, cộng dần | `O(n√n)` | `O(1)` | đối chiếu |
| 2. Cửa sổ trượt | hai con trỏ trên `1..n` | `O(n)` | `O(1)` | `n ≤ 10⁶`, bài tổng đoạn tổng quát |
| 3. Duyệt `k` | `k | n − k(k−1)/2` | `O(√n)` | `O(1)` | **nộp bài** |
| 4. Đếm ước lẻ | song ánh dãy ↔ ước lẻ | `O(√n)` | `O(1)` | khi biết kết quả số học |
| 5. Phân tích thừa số | `∏(aᵢ+1)` với `pᵢ` lẻ | `O(√n)` xấu nhất | `O(1)` | nhanh nhất thực tế |

## 17. Bài tập

**Bài 1 (đoán output).** `consecutiveNumbersSum(16)`? `(21)`? `(100)`?

**Bài 2 (tính tay).** Liệt kê mọi cách viết `45` thành tổng các số dương liên tiếp và đối chiếu với các ước lẻ của 45.

**Bài 3 (điền TODO).** Viết `ways_list_by_divisors(n)` sinh trực tiếp các dãy từ các ước lẻ (dùng `A = min(d, 2n/d)`, `B = max(...)`).

**Bài 4 (tự viết lại).** Viết lại cách 3 với vòng `for k in range(1, isqrt(2n) + 2)` và điều kiện rõ ràng.

**Bài 5 (mini challenge).** Tìm số `n <= 10⁵` có **nhiều cách nhất**, dùng sàng để đếm ước lẻ cho mọi `n` một lượt.

In [30]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def ways_list_by_divisors_todo(n):
    # TODO: với mỗi ước lẻ d của n: e = 2n // d; A, B = min(d, e), max(d, e); k = A; x = (B - A + 1)//2
    pass

# Bài 4 — TODO
def consec_for_todo(n):
    # TODO: for k in range(1, isqrt(2*n) + 2): if k*(k+1)//2 > n: break ...
    pass

# Bài 5 — TODO
def most_ways_todo(limit):
    # TODO: cnt = [0]*(limit+1); for d lẻ: for bội của d: cnt[m] += 1
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: tìm phần lẻ của `n`.
- Mức 2: `16 → 1`, `21 = 3·7 → 4 ước lẻ`, `100 = 4·25 → ước lẻ của 25`.
- Mức 3: `1`, `4`, `3`.

**Bài 2.**
- Mức 1: ước lẻ của 45: `1, 3, 5, 9, 15, 45` → 6 cách.
- Mức 2: duyệt `k = 1..9` (`9·10/2 = 45`).
- Mức 3: `45; 22+23; 14+15+16; 7..11; 5..10; 1..9`.

**Bài 3.**
- Mức 1: `2n = d · e`.
- Mức 2: một trong `d`, `e` chẵn, một lẻ, nên khác nhau.
- Mức 3: `k = min`, `x = (max − min + 1) / 2`.

**Bài 4.**
- Mức 1: `k(k+1)/2 <= n` ⇒ `k <= √(2n)`.
- Mức 2: thêm `break` khi vượt.
- Mức 3: kiểm tra với `Solution` trên `1..2000`.

**Bài 5.**
- Mức 1: sàng kiểu Eratosthenes trên các ước lẻ: `O(n log n)`.
- Mức 2: lưu `argmax`.
- Mức 3: kiểm tra vài giá trị bằng `Solution`.

## 19. Đáp án tham khảo

In [31]:
from math import isqrt

# Bài 1
assert [Solution().consecutiveNumbersSum(x) for x in (16, 21, 100)] == [1, 4, 3]

# Bài 2
ways45 = list_ways(45)
assert len(ways45) == 6 == len([d for d in divisors(45) if d % 2])
print([w if len(w) <= 3 else f"{w[0]}..{w[-1]}" for w in ways45])

# Bài 3
def ways_list_by_divisors(n):
    res = []
    for d in divisors(n):
        if d % 2 == 0:
            continue
        e = 2 * n // d
        A, B = min(d, e), max(d, e)
        k, x = A, (B - A + 1) // 2
        res.append(list(range(x, x + k)))
    return sorted(res, key=len)

for n in range(1, 300):
    assert sorted(map(tuple, ways_list_by_divisors(n))) == sorted(map(tuple, list_ways(n)))

# Bài 4
def consec_for(n):
    count = 0
    for k in range(1, isqrt(2 * n) + 2):
        if k * (k + 1) // 2 > n:
            break
        if (n - k * (k - 1) // 2) % k == 0:
            count += 1
    return count

assert all(consec_for(n) == Solution().consecutiveNumbersSum(n) for n in range(1, 2001))

# Bài 5
def most_ways(limit):
    cnt = [0] * (limit + 1)
    for d in range(1, limit + 1, 2):
        for m in range(d, limit + 1, d):
            cnt[m] += 1
    best = max(range(1, limit + 1), key=lambda m: cnt[m])
    return best, cnt[best]

best_n, best_c = most_ways(10 ** 5)
assert Solution().consecutiveNumbersSum(best_n) == best_c
print("n <= 1e5 nhiều cách nhất:", best_n, "với", best_c, "cách")
print("Tất cả đáp án đúng")

[[45], [22, 23], [14, 15, 16], '7..11', '5..10', '1..9']
n <= 1e5 nhiều cách nhất: 45045 với 48 cách
Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (duyệt số số hạng):**

```python
count, k, m = 0, 1, n          # m = n - k(k-1)/2
while m >= k:                  # x = m/k >= 1
    if m % k == 0: count += 1
    m -= k; k += 1
return count
```

**Template (ước lẻ):**

```python
while n % 2 == 0: n //= 2
res, p = 1, 3
while p * p <= n:
    a = 0
    while n % p == 0: n //= p; a += 1
    res *= a + 1; p += 2
if n > 1: res *= 2
return res
```

**Checklist trước khi submit:**

- [ ] Số hạng phải **dương**: điều kiện `k(k+1)/2 <= n`, không phải `k(k−1)/2 <= n`.
- [ ] Chỉ dùng số nguyên (`//`, `%`), không dùng `/`.
- [ ] Tính cả `k = 1` (chính `n`).
- [ ] Với cách ước lẻ: bỏ thừa số 2, nhớ thừa số nguyên tố lớn còn lại.
- [ ] Thử nhanh `5 → 2`, `9 → 3`, `15 → 4`, `8 → 1`.